# 🧬 Long Short-Term Memory (LSTM) — Solutions Notebook

**This notebook contains complete, verified solutions.**

**Difficulty**: ⭐⭐ Intermediate  
**Time**: ~60 minutes

---


## 🎯 Section 1: Overview

A **Long Short-Term Memory (LSTM)** network is an improved variant of recurrent neural networks. It introduces a **cell state** ($C_t$) that acts as a linear conveyor belt, allowing gradient signals to propagate over long sequences without scaling exponentially.

### Gating Mechanism
LSTM uses three gates to regulate input/output flows: the forget gate, the input gate, and the output gate.


## 📐 Section 2: Math & Intuition

### LSTM Gate Equations
Given input $x_t$ and previous hidden state $h_{t-1}$:
1. **Forget Gate**: $f_t = \sigma(x_t W_{xf} + h_{t-1} W_{hf} + b_f)$
2. **Input Gate**: $i_t = \sigma(x_t W_{xi} + h_{t-1} W_{hi} + b_i)$
3. **Candidate Cell State**: $\tilde{C}_t = \tanh(x_t W_{xc} + h_{t-1} W_{hc} + b_c)$
4. **Cell State Update**: $C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$
5. **Output Gate**: $o_t = \sigma(x_t W_{xo} + h_{t-1} W_{ho} + b_o)$
6. **Hidden State**: $h_t = o_t * \tanh(C_t)$

where $\sigma$ is the element-wise sigmoid activation.


## 🔧 Section 3: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('LSTM Setup complete! ✅')


### 3.1 LSTM Cell Forward Pass


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def lstm_cell_forward(xt, h_prev, c_prev, W, b):
    """
    xt: input vector (batch_size, input_dim)
    h_prev: previous hidden state (batch_size, hidden_dim)
    c_prev: previous cell state (batch_size, hidden_dim)
    W: unified weights containing stacked weights for [f, i, c, o]
       shape is (input_dim + hidden_dim, 4 * hidden_dim)
    b: bias vector of shape (1, 4 * hidden_dim)
    """
    batch_size, hidden_dim = h_prev.shape
    
    # Concatenate inputs
    concat = np.hstack((xt, h_prev))  # shape: (batch_size, input_dim + hidden_dim)
    
    # Calculate linear combination
    A = concat @ W + b  # shape: (batch_size, 4 * hidden_dim)
    
    # Slice gates
    f_gate = A[:, :hidden_dim]
    i_gate = A[:, hidden_dim:2*hidden_dim]
    c_cand = A[:, 2*hidden_dim:3*hidden_dim]
    o_gate = A[:, 3*hidden_dim:]
    
    f = sigmoid(f_gate)
    i = sigmoid(i_gate)
    c_bar = np.tanh(c_cand)
    o = sigmoid(o_gate)
    
    c_next = f * c_prev + i * c_bar
    h_next = o * np.tanh(c_next)
    
    return h_next, c_next


### 3.2 Verify LSTM Cell


In [ ]:
xt_v = np.random.randn(2, 3)     # batch_size=2, input_dim=3
h_v = np.zeros((2, 4))           # hidden_dim=4
c_v = np.zeros((2, 4))
W_v = np.random.randn(7, 16)     # (3+4) input+hidden, 4*4 gates
b_v = np.zeros((1, 16))

h_n, c_n = lstm_cell_forward(xt_v, h_v, c_v, W_v, b_v)
print('Hidden state output shape (should be [2, 4]):', list(h_n.shape))
if 'TODO' not in lstm_cell_forward.__code__.co_consts:
    assert list(h_n.shape) == [2, 4]
    assert list(c_n.shape) == [2, 4]
    print('LSTM Cell forward pass verified! ✅')


## 📦 Section 4: Library Implementation


In [ ]:
import torch
import torch.nn as nn

class PyTorchLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        # Take the output of the last time step
        return self.fc(out[:, -1, :])
        


In [ ]:
model = PyTorchLSTM(input_dim=4, hidden_dim=8, output_dim=1)
dummy_batch = torch.randn(5, 10, 4)  # batch size of 5, seq len of 10, input dims of 4
out = model(dummy_batch)
print('Output shape (should be [5, 1]):', list(out.shape))
assert list(out.shape) == [5, 1]
print('PyTorch LSTM check passed! ✅')


## 🧪 Section 5: Experiments


Compare LSTM vs. standard RNN on a simple memory task. The target is simply the first character in the sequence.


In [ ]:
# Generate memory data: sequences of length 40, classes based on index 0
N = 200
seq_len = 40
X_mem = np.random.randn(N, seq_len, 1)
# Class 1 if first step > 0, else Class 0
y_mem = (X_mem[:, 0, 0] > 0).astype(int)

X_mem_t = torch.FloatTensor(X_mem)
y_mem_t = torch.FloatTensor(y_mem).view(-1, 1)

# Training LSTM vs RNN
class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 8, batch_first=True)
        self.fc = nn.Linear(8, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return torch.sigmoid(self.fc(out[:, -1, :]))

class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 8, batch_first=True)
        self.fc = nn.Linear(8, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return torch.sigmoid(self.fc(out[:, -1, :]))

models = {'RNN': SimpleRNN(), 'LSTM': SimpleLSTM()}
histories = {}

for name, m in models.items():
    opt = torch.optim.Adam(m.parameters(), lr=0.02)
    crit = nn.BCELoss()
    losses = []
    for epoch in range(120):
        opt.zero_grad()
        pred = m(X_mem_t)
        l = crit(pred, y_mem_t)
        l.backward()
        opt.step()
        losses.append(l.item())
    histories[name] = losses

plt.figure(figsize=(8, 5))
for name, losses in histories.items():
    plt.plot(losses, label=name)
plt.title('Memory Task Convergence (Sequence Length = 40)')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.legend()
plt.show()


## ❓ Section 6: Interview Questions


### Q1: Explain the role of the cell state ($C_t$) in LSTMs.
**Answer**:
The cell state $C_t$ serves as the memory pathway of the LSTM cell. It has only linear operations (multiplication by forget gate $f_t$ and addition of input candidate $i_t * \tilde{C}_t$). Because this path is linear, error signals backpropagating along it can flow backward across long sequence steps without undergoing exponential growth or decay, resolving vanishing gradients.

### Q2: Walk through each of the gates in an LSTM cell.
**Answer**:
- **Forget Gate ($f_t$)**: Outputs values in $[0, 1]$ via Sigmoid, determining how much of the historical cell state $C_{t-1}$ to retain.
- **Input Gate ($i_t$)**: Decides which new input coordinates to update in our memory.
- **Candidate Cell State ($\tilde{C}_t$)**: Outputs values in $[-1, 1]$ via Tanh, creating candidate values to append to the cell state.
- **Output Gate ($o_t$)**: Determines which parts of the updated cell state to write into the output hidden state $h_t$.

### Q3: Why does LSTM avoid the vanishing gradient problem?
**Answer**:
In Vanilla RNNs, backpropagating through time requires multiplying by $W_{hh}^T$ at each step. In LSTMs, the gradient flow along the cell state $C_t$ path is controlled by addition and scaling by $f_t$. If the network learns to keep the forget gate $f_t \approx 1.0$, the gradient is propagated back through time nearly unimpeded, preventing the signal from vanishing.

### Q4: What is the difference between LSTM cell state and hidden state?
**Answer**:
- **Cell State ($C_t$)**: Represents the internal long-term memory of the unit. It is not exposed to other layers directly.
- **Hidden State ($h_t$)**: Represents the external short-term memory / output of the cell at step $t$. It is computed by taking a non-linear activation of the cell state gated by the output gate.


## 🏆 Section 7: Challenge — Gating Calculus


**Challenge**: Derive the gradient of the next cell state ($C_t$) with respect to the forget gate ($f_t$).


**Mathematical Solution**:
The LSTM update equation for the cell state is:
$$C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$$
Taking the partial derivative of $C_t$ with respect to $f_t$ (treating other variables as constant local variables at step $t$):
$$\frac{\partial C_t}{\partial f_t} = C_{t-1}$$
This simple derivative shows that the gradient scale directly corresponds to the magnitude of the previous cell state.
